<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ICS40125 - Laboratorio N°03





**Objetivo**: Aplicar técnicas avanzadas de manipulación y análisis de datos con pandas sobre un conjunto real de datos de contenido de Netflix, reforzando buenas prácticas y métodos eficientes sin recurrir a `groupby`, `merge`, `pivot`, ni `join`.



**Dataset**:

Trabajaremos con el archivo `netflix_titles.csv`, que contiene información sobre los títulos disponibles en la plataforma Netflix hasta el año 2021.

| Variable       | Clase     | Descripción                                                                 |
|----------------|-----------|------------------------------------------------------------------------------|
| show_id        | caracter  | Identificador único del título en el catálogo de Netflix.                   |
| type           | caracter  | Tipo de contenido: 'Movie' o 'TV Show'.                                     |
| title          | caracter  | Título del contenido.                                                       |
| director       | caracter  | Nombre del director (puede ser nulo).                                       |
| cast           | caracter  | Lista de actores principales (puede ser nulo).                              |
| country        | caracter  | País o países donde se produjo el contenido.                                |
| date_added     | fecha     | Fecha en la que el título fue agregado al catálogo de Netflix.              |
| release_year   | entero    | Año de lanzamiento original del título.                                     |
| rating         | caracter  | Clasificación por edad (por ejemplo: 'PG-13', 'TV-MA').                      |
| duration       | caracter  | Duración del contenido (minutos o número de temporadas para series).        |
| listed_in      | caracter  | Categorías o géneros en los que está clasificado el contenido.              |
| description    | caracter  | Breve sinopsis del contenido.                                               |




In [ ]:
import pandas as pd

# Cargar datos
df = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...



### Parte 1: Limpieza y preparación

1. Revisar y describir el dataset:

   * ¿Cuántas filas y columnas tiene?
   * ¿Qué tipos de datos hay?
   * ¿Cuántos valores nulos hay por columna?

2. Transformar la columna `date_added` a tipo fecha.

3. Crear columnas auxiliares con `assign`:

   * Año (`year_added`)
   * Mes (`month_added`)



In [2]:
import pandas as pd

# Cargar dataset (ajusta el nombre si es necesario)
df = pd.read_csv("https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/netflix_titles.csv")

# 1. Revisar dataset
print("Filas y columnas:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print("\nValores nulos por columna:")
print(df.isnull().sum())

# 2. Transformar date_added a fecha
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")

# 3. Crear columnas auxiliares
df = df.assign(
    year_added = df["date_added"].dt.year,
    month_added = df["date_added"].dt.month
)

print(df.head())

Filas y columnas: (8807, 12)

Tipos de datos:
show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object
description     object
dtype: object

Valores nulos por columna:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64
  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water              NaN   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans              NaN   
4      s5  TV Show           Kota Factory              NaN   

In [4]:
# 4. Seleccionar películas (type == 'Movie') que fueron agregadas después del año 2018.
movies_added_after_2018 = df.loc[(df["type"] == "Movie") & (df["year_added"] > 2018)]
print("--- Movies added after 2018 (first 5 rows) ---")
print(movies_added_after_2018.head())
print(f"Total movies added after 2018: {len(movies_added_after_2018)}")

# 5. Filtrar títulos que contienen la palabra 'love' (sin distinguir mayúsculas/minúsculas).
love_titles = df[df["title"].str.contains("love", case=False, na=False)]
print("\n--- Titles containing 'love' (first 5 rows) ---")
print(love_titles.head())
print(f"Total titles containing 'love': {len(love_titles)}")


# Extraer la duración en minutos para las películas desde la columna `duration`.
# Initialize the column with NaN (or pd.NA for newer pandas)
df["duration_int"] = pd.NA
# Apply extraction only for movies and convert to float
df.loc[df["type"] == "Movie", "duration_int"] = df["duration"].str.extract("(\\d+)", expand=False).astype(float)
print("\n--- DataFrame with 'duration_int' for movies (first 5 rows) ---")
print(df[['type', 'duration', 'duration_int']].head())
print("\n--- Sample of TV Shows to show NaN in 'duration_int' ---")
print(df[df['type'] == 'TV Show'][['type', 'duration', 'duration_int']].head())


# 6. Aplicar explode() sobre la columna listed_in para obtener una fila por cada género.
# Ensure 'listed_in' is a list of strings before exploding, in case the cell is run independently.
if not isinstance(df["listed_in"].iloc[0], list):
    df["listed_in"] = df["listed_in"].str.split(", ")
df_exploded = df.explode("listed_in")

print("\n--- df_exploded (first 5 rows) ---")
print(df_exploded.head())

--- Movies added after 2018 (first 5 rows) ---
   show_id   type                             title  \
0       s1  Movie              Dick Johnson Is Dead   
6       s7  Movie  My Little Pony: A New Generation   
7       s8  Movie                           Sankofa   
9      s10  Movie                      The Starling   
12     s13  Movie                      Je Suis Karl   

                         director  \
0                 Kirsten Johnson   
6   Robert Cullen, José Luis Ucha   
7                    Haile Gerima   
9                  Theodore Melfi   
12            Christian Schwochow   

                                                 cast  \
0                                                 NaN   
6   Vanessa Hudgens, Kimiko Glenn, James Marsden, ...   
7   Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...   
9   Melissa McCarthy, Chris O'Dowd, Kevin Kline, T...   
12  Luna Wedler, Jannis Niewöhner, Milan Peschel, ...   

                                              country da

## Parte 2: Técnicas avanzadas de pandas

4. Utilizar `.loc` para seleccionar películas (`type == 'Movie'`) que fueron agregadas después del año 2018.

5. Utilizar `str.contains()` y `str.extract()`:

   * Filtrar títulos que contienen la palabra 'love' (sin distinguir mayúsculas/minúsculas).
   * Extraer la duración en minutos para las películas desde la columna `duration`.

6. Aplicar `explode()` sobre la columna `listed_in` para obtener una fila por cada género.

7. Obtener un top 10 de géneros más frecuentes utilizando `value_counts()`.

8. Aplicar `where()` y `mask()` para marcar las películas de más de 120 minutos como contenido largo en una nueva columna.

9. Utilizar `.loc` para filtrar películas que cumplen con:

   * Más de 100 minutos de duración.
   * Rating igual a `'R'`.
   * País igual a `'United States'`.

10. Utilizar `.style` para formatear visualmente el top 10 de películas más largas.

In [6]:
# 7. Obtener un top 10 de géneros más frecuentes utilizando value_counts().
genre_counts = df_exploded['listed_in'].value_counts()
top_10_genres = genre_counts.head(10)
print("\n--- Top 10 Géneros más frecuentes ---")
print(top_10_genres)

# 8. Aplicar where() y mask() para marcar las películas de más de 120 minutos como contenido largo en una nueva columna.
# Creamos una copia para evitar SettingWithCopyWarning y asegurar que 'df' se modifica
df_copy = df.copy()
df_copy['is_long_movie'] = df_copy['duration_int'].apply(lambda x: 'Desconocido' if pd.isna(x) else ('Largo' if x > 120 else 'Corto'))
# Displaying an example (movies only to show effect)
print("\n--- Muestra de películas con 'is_long_movie' ---")
print(df_copy[df_copy['type'] == 'Movie'][['title', 'duration_int', 'is_long_movie']].head(10))
df = df_copy # Update the original df with the new column

# 9. Utilizar .loc para filtrar películas que cumplen con:
#    * Más de 100 minutos de duración.
#    * Rating igual a 'R'.
#    * País igual a 'United States'.
filtered_movies = df.loc[
    (df['type'] == 'Movie') &
    (df['duration_int'] > 100) &
    (df['rating'] == 'R') &
    (df['country'].str.contains('United States', na=False))
]
print("\n--- Películas filtradas (duración > 100, rating 'R', país 'United States') ---")
print(filtered_movies[['title', 'duration_int', 'rating', 'country']].head())
print(f"Total de películas que cumplen los criterios: {len(filtered_movies)}")

# 10. Utilizar .style para formatear visualmente el top 10 de películas más largas.
top_10_longest_movies = df[df['type'] == 'Movie'].sort_values(by='duration_int', ascending=False).head(10)
print("\n--- Top 10 películas más largas (formato con .style) ---")
display(top_10_longest_movies[['title', 'duration_int', 'rating', 'country']].style.background_gradient(cmap='Blues'))


--- Top 10 Géneros más frecuentes ---
listed_in
International Movies        2752
Dramas                      2427
Comedies                    1674
International TV Shows      1351
Documentaries                869
Action & Adventure           859
TV Dramas                    763
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Name: count, dtype: int64

--- Muestra de películas con 'is_long_movie' ---
                                                title duration_int  \
0                                Dick Johnson Is Dead         90.0   
6                    My Little Pony: A New Generation         91.0   
7                                             Sankofa        125.0   
9                                        The Starling        104.0   
12                                       Je Suis Karl        127.0   
13                   Confessions of an Invisible Girl         91.0   
16  Europe's Most Dangerous Man: Otto Skorzeny in ...  

,title,duration_int,rating,country
4253,Black Mirror: Bandersnatch,312.000000,TV-MA,United States
717,Headspace: Unwind Your Mind,273.000000,TV-G,nan
2491,The School of Mischief,253.000000,TV-14,Egypt
2487,No Longer kids,237.000000,TV-14,Egypt
2484,Lock Your Girls In,233.000000,TV-PG,nan
2488,Raya and Sakina,230.000000,TV-14,nan
166,Once Upon a Time in America,229.000000,R,"Italy, United States"
7932,Sangam,228.000000,TV-14,India
1019,Lagaan,224.000000,PG,"India, United Kingdom"
4573,Jodhaa Akbar,214.000000,TV-14,India




### Pregunta Desafío

11. ¿Cuáles son las combinaciones más frecuentes de género y rating en el dataset?
    (Sugerencia: utilizar `value_counts` con `subset=["genre", "rating"]` después de aplicar `explode()`).



### Bonus: Análisis de duplicados y limpieza

12. ¿Existen películas con el mismo nombre (`title`) pero con distinto año de lanzamiento (`release_year`)?
13. ¿Cuántos títulos únicos hay en total en la columna `title`?





In [8]:
# 11. ¿Cuáles son las combinaciones más frecuentes de género y rating en el dataset?
# Sugerencia: utilizar value_counts con subset=["genre", "rating"] después de aplicar explode().

# Asegurarse de que 'rating' está en df_exploded si no lo estuviera ya
# (ya lo está porque df_exploded se crea a partir de df que tiene 'rating')
genre_rating_combinations = df_exploded[['listed_in', 'rating']].value_counts().head(10)
print("\n--- Top 10 combinaciones más frecuentes de Género y Rating ---")
print(genre_rating_combinations)

# 12. ¿Existen películas con el mismo nombre (title) pero con distinto año de lanzamiento (release_year)?
# Primero, identificamos los títulos que aparecen con más de un año de lanzamiento único.
titles_with_multiple_release_years = df.groupby('title')['release_year'].nunique()
# Filtramos para encontrar aquellos títulos donde el conteo de años únicos es mayor a 1
duplicate_titles_different_years = titles_with_multiple_release_years[titles_with_multiple_release_years > 1]

if not duplicate_titles_different_years.empty:
    print("\n--- Títulos con el mismo nombre pero diferente año de lanzamiento ---")
    # Para ver los detalles de estas películas, filtramos el DataFrame original
    titles_to_show = duplicate_titles_different_years.index.tolist()
    display(df[df['title'].isin(titles_to_show)][['title', 'release_year', 'type', 'date_added']].sort_values(by='title'))
else:
    print("\n--- No se encontraron títulos con el mismo nombre pero diferente año de lanzamiento ---")

# 13. ¿Cuántos títulos únicos hay en total en la columna `title`?
unique_titles_count = df['title'].nunique()
print(f"\n--- Número total de títulos únicos: {unique_titles_count} ---")



--- Top 10 combinaciones más frecuentes de Género y Rating ---
listed_in               rating
International Movies    TV-MA     1130
                        TV-14     1065
Dramas                  TV-MA      830
International TV Shows  TV-MA      714
Dramas                  TV-14      693
International TV Shows  TV-14      472
Comedies                TV-14      465
TV Dramas               TV-MA      434
Comedies                TV-MA      431
Dramas                  R          375
Name: count, dtype: int64

--- No se encontraron títulos con el mismo nombre pero diferente año de lanzamiento ---

--- Número total de títulos únicos: 8807 ---
